# 4. Performance Tracking

## Overview
This notebook implements performance tracking for our ETF portfolio across all four asset classes, including:
1. Historical price data collection via yfinance
2. Return calculations (monthly, annual, YTD, MTD)
3. Risk metrics (volatility, Sharpe ratio, max drawdown)
4. Portfolio-level P&L and performance reporting

Portfolio data is loaded from the  table in the SQLite database (year = 2026), falling back to .

## Required Libraries

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.database import load_portfolio
from etf_utils.metrics import (
    calculate_annualized_volatility,
    calculate_period_metrics,
    calculate_daily_pnl,
)

provider = DataProvider()

# Load portfolio from DB (year=2026); falls back to CSV if DB is empty
portfolio_df = load_portfolio(year=2025)
if portfolio_df.empty:
    portfolio_df = pd.read_csv(DATA_OUTPUT / 'final_portfolio.csv')
print(f"Loaded {len(portfolio_df)} positions from 2025 portfolio "
      f"(asset classes: {portfolio_df['asset_class'].unique().tolist()})")

Loaded 16 positions from 2026 portfolio (asset classes: ['equity', 'bonds'])


## ETF Performance Calculator
Using your existing code to calculate ETF metrics:

In [6]:
def calculate_etf_metrics(symbol, years=range(2021, 2026)):
    """Calculate annual performance metrics for an ETF using DataProvider."""
    df = provider.get_historical_prices(symbol)
    annual_results = []

    for year in years:
        df_year = df[df.index.year == year]
        if len(df_year) < 2:
            continue

        start_price = df_year["close"].iloc[0]
        end_price = df_year["close"].iloc[-1]
        annual_return = ((end_price / start_price) - 1) * 100
        volatility = calculate_annualized_volatility(df_year["close"]) * 100

        annual_results.append({
            "Year": year,
            "Start Price": round(float(start_price), 2),
            "End Price": round(float(end_price), 2),
            "Annualized Return (%)": round(float(annual_return), 2),
            "Volatility (%)": round(float(volatility), 2),
        })

    return pd.DataFrame(annual_results)

## Annual Performance Metrics

## Portfolio Performance
Calculate weighted portfolio performance:

## Calculate Portfolio Performance
First, we'll calculate the performance metrics for each ETF and combine them into portfolio-level metrics:

In [7]:
# Initialize portfolio performance tracking
portfolio_performance = pd.DataFrame()

# Calculate performance for each ETF using DataProvider (no API key needed)
for idx, row in portfolio_df.iterrows():
    try:
        etf_metrics = calculate_etf_metrics(row['ticker'])
        if not etf_metrics.empty:
            weight = row['investment'] / portfolio_df['investment'].sum()
            etf_metrics['Weighted Return (%)'] = etf_metrics['Annualized Return (%)'] * weight
            etf_metrics['Weighted Volatility (%)'] = etf_metrics['Volatility (%)'] * weight
            etf_metrics['ETF'] = row['ticker']
            portfolio_performance = pd.concat([portfolio_performance, etf_metrics])
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

# Calculate portfolio-level metrics
if not portfolio_performance.empty:
    portfolio_summary = pd.DataFrame({
        'Year': portfolio_performance['Year'].unique(),
        'Return (%)': portfolio_performance.groupby('Year')['Weighted Return (%)'].sum().values,
        'Volatility (%)': portfolio_performance.groupby('Year')['Weighted Volatility (%)'].sum().values,
    })

    print("\nPortfolio Performance Summary:")
    display(portfolio_summary)
else:
    print("No performance data available")

No performance data available


In [4]:
# Calculate risk-adjusted metrics from portfolio summary
from etf_utils.metrics import calculate_sharpe_ratio

if 'portfolio_summary' in locals() and not portfolio_summary.empty:
    risk_free_rate = 4.0  # UK base rate ~4% in 2024
    portfolio_summary['Sharpe Ratio'] = portfolio_summary.apply(
        lambda row: round(calculate_sharpe_ratio(
            row['Return (%)'], row['Volatility (%)'], risk_free_rate
        ), 2),
        axis=1,
    )
    print("\nRisk-Adjusted Performance Metrics:")
    display(portfolio_summary)
else:
    print("Unable to calculate risk metrics - no performance data available")

Unable to calculate risk metrics - no performance data available


## Recent Performance Metrics (YTD and MTD)
Calculate performance metrics for:
1. Year to Date (2026)
2. Month to Date (March 2026)

In [5]:
# Calculate YTD and MTD performance using DataProvider
ytd_start = "2025-01-01"
mtd_start = "2025-02-01"

ytd_performance = []
mtd_performance = []

for idx, row in portfolio_df.iterrows():
    try:
        df = provider.get_historical_prices(row['ticker'])
        weight = row['investment'] / portfolio_df['investment'].sum()

        # YTD metrics
        ytd = calculate_period_metrics(df, ytd_start)
        if ytd and not np.isnan(ytd['return']):
            ytd_performance.append({
                'ETF': row['ticker'],
                'Return (%)': round(ytd['return'] * 100, 2),
                'Volatility (%)': round(ytd['volatility'] * 100, 2),
                'Weight': weight,
                'Weighted Return (%)': round(ytd['return'] * 100 * weight, 2),
                'Weighted Volatility (%)': round(ytd['volatility'] * 100 * weight, 2),
            })

        # MTD metrics
        mtd = calculate_period_metrics(df, mtd_start)
        if mtd and not np.isnan(mtd['return']):
            mtd_performance.append({
                'ETF': row['ticker'],
                'Return (%)': round(mtd['return'] * 100, 2),
                'Volatility (%)': round(mtd['volatility'] * 100, 2),
                'Weight': weight,
                'Weighted Return (%)': round(mtd['return'] * 100 * weight, 2),
                'Weighted Volatility (%)': round(mtd['volatility'] * 100 * weight, 2),
            })
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

ytd_df = pd.DataFrame(ytd_performance)
mtd_df = pd.DataFrame(mtd_performance)

if not ytd_df.empty:
    print("\nYear to Date (2025) Portfolio Performance:")
    print(f"Total Return: {ytd_df['Weighted Return (%)'].sum():.2f}%")
    print(f"Portfolio Volatility: {ytd_df['Weighted Volatility (%)'].sum():.2f}%")
    print("\nTop 3 YTD Contributors:")
    display(ytd_df.nlargest(3, 'Weighted Return (%)')[['ETF', 'Return (%)', 'Weighted Return (%)']])

if not mtd_df.empty:
    print("\nMonth to Date Portfolio Performance:")
    print(f"Total Return: {mtd_df['Weighted Return (%)'].sum():.2f}%")
    print(f"Portfolio Volatility: {mtd_df['Weighted Volatility (%)'].sum():.2f}%")

# YTD Sharpe ratio
risk_free_rate = 0.04
if not ytd_df.empty:
    ytd_return = ytd_df['Weighted Return (%)'].sum() / 100
    ytd_vol = ytd_df['Weighted Volatility (%)'].sum() / 100
    ytd_sharpe = (ytd_return - risk_free_rate * 0.5) / ytd_vol if ytd_vol > 0 else float('nan')
    print(f"\nYTD Sharpe Ratio: {ytd_sharpe:.2f}")

C:\Users\rakes\AppData\Local\Temp\ipykernel_34808\1660607534.py:14: UserWarning: Fewer than 2 data points in [2025-01-01, None] — returning NaN.
  ytd = calculate_period_metrics(df, ytd_start)
C:\Users\rakes\AppData\Local\Temp\ipykernel_34808\1660607534.py:26: UserWarning: Fewer than 2 data points in [2025-02-01, None] — returning NaN.
  mtd = calculate_period_metrics(df, mtd_start)



Year to Date (2025) Portfolio Performance:
Total Return: -2.16%
Portfolio Volatility: 5906.19%

Top 3 YTD Contributors:


,ETF,Return (%),Weighted Return (%)
2,QYLP,3.45,0.24
10,UC81,1.99,0.02
11,EMCP,1.90,0.02



Month to Date Portfolio Performance:
Total Return: -2.16%
Portfolio Volatility: 5906.19%

YTD Sharpe Ratio: -0.00


In [ ]:
# Calculate daily PnL for the portfolio using DataProvider
pnl_start = "2025-02-01"

daily_pnl_list = []

for idx, row in portfolio_df.iterrows():
    try:
        df = provider.get_historical_prices(row['ticker'])
        pnl_df = calculate_daily_pnl(df, row['investment'], pnl_start)

        if pnl_df is not None and not pnl_df.empty:
            weight = row['investment'] / portfolio_df['investment'].sum()
            pnl_df = pnl_df.copy()
            pnl_df['ETF'] = row['ticker']
            pnl_df['Weight'] = weight
            pnl_df['Weighted PnL'] = pnl_df['pnl'] * weight
            daily_pnl_list.append(pnl_df)
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

if daily_pnl_list:
    daily_pnl = pd.concat(daily_pnl_list)
    print("\nDaily PnL Summary by ETF:")
    display(daily_pnl.groupby('ETF')['Weighted PnL'].sum().sort_values(ascending=False))

    # Portfolio-level daily PnL
    portfolio_daily = daily_pnl.groupby(daily_pnl.index)['Weighted PnL'].sum()
    print(f"\nTotal Portfolio PnL since {pnl_start}: {portfolio_daily.sum():.2f}")
else:
    print("No PnL data available")

In [ ]:
import plotly.graph_objects as go

# Plot daily returns for a selected ETF
selected_etf = "AUAD"
etf_data = daily_pnl[daily_pnl['ETF'] == selected_etf].copy()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=etf_data.index,
        y=etf_data['daily_return'],
        mode='lines+markers',
        name='Daily Return',
        line=dict(color='blue'),
        hovertemplate='Date: %{x}<br>Return: %{y:.2%}<extra></extra>'
    )
)

fig.update_layout(
    title=f'{selected_etf} ETF Daily Returns',
    xaxis_title='Date',
    yaxis_title='Daily Return',
    yaxis_tickformat='.2%',
    hovermode='x unified',
    template='plotly_white',
    width=1000,
    height=600
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.show()

In [ ]:
# Show recent data for selected ETF
etf_data.tail(10)

In [ ]:
# Portfolio-level daily PnL
portfolio_daily_pnl = daily_pnl.groupby(daily_pnl.index)['Weighted PnL'].sum()
portfolio_daily_pnl.tail(10)